# Week 5 on a Mac: RAG with local models

The course notebooks (`day1.ipynb` ... `day5.ipynb`) call `gpt-4.1-nano` and OpenAI's `text-embedding-3-large`,
and the modules they import (`implementation/`, `pro_implementation/`, `evaluation/eval.py`) build OpenAI clients
as soon as they're imported - so without an API key nothing in this week runs. This notebook goes through the same
five days with everything on the Mac, using [local_rag.py](local_rag.py):

| course | here |
|--------|------|
| gpt-4.1-nano | `qwen2.5:14b` on Ollama (native `/api/chat` with `num_ctx=16384`) |
| litellm `response_format=Model` | Ollama's `format` parameter with the pydantic model's JSON schema |
| text-embedding-3-large | `all-MiniLM-L6-v2` (day 2) and `BAAI/bge-base-en-v1.5` via sentence-transformers |
| LLM reranker | the same LLM reranker, plus a local cross-encoder as a faster alternative |

The vector stores live in `week5/local_db/` (gitignored). Start Ollama first (`ollama serve`).

The week's exercise - the Spanish tutor answering from its own notes - is at the end and in
[spanish_tutor_rag.py](spanish_tutor_rag.py).

In [1]:
import time
from collections import Counter

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import tiktoken
from sklearn.manifold import TSNE

import local_rag as rag_lib
from local_rag import RAG, fetch_documents, llm

pd.set_option("display.max_colwidth", 120)
print("Ollama running:", rag_lib.ollama_running(), "| chat model:", rag_lib.CHAT_MODEL)

Ollama running: True | chat model: qwen2.5:14b


## Day 1 – brute-force "RAG": look names up in a dictionary

Every employee and product file goes into a dict keyed by surname or product name. If a word of the question is a
key, that whole file is pasted into the system prompt.

In [2]:
knowledge = {}
for document in fetch_documents():
    name = document["source"].rsplit("/", 1)[-1].removesuffix(".md")
    if document["type"] == "employees":
        knowledge[name.split(" ")[-1].lower()] = document["text"]
    elif document["type"] == "products":
        knowledge[name.lower()] = document["text"]

print(len(knowledge), "entries:", ", ".join(sorted(knowledge)))

38 entries: adams, anderson, bishop, bizllm, blake, brooks, carllm, carter, chen, claimllm, foster, greene, harper, healthllm, homellm, johnson, kim, lancaster, lifellm, liu, markellm, martinez, o'brien, park, patel, rellm, rivera, rodriguez, sharma, spencer, thompson, thomson, tran, trenton, walker, williams, wilson, zhang


In [3]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""


def get_relevant_context(message):
    text = "".join(ch for ch in message if ch.isalpha() or ch.isspace())
    return [knowledge[word] for word in text.lower().split() if word in knowledge]


def additional_context(message):
    relevant = get_relevant_context(message)
    if not relevant:
        return "There is no additional context relevant to the user's question."
    return "The following additional context might be relevant in answering the user's question:\n\n" + "\n\n".join(relevant)


def day1_chat(message, history):
    history = [{"role": m["role"], "content": m["content"]} for m in history]
    messages = [{"role": "system", "content": SYSTEM_PREFIX + additional_context(message)}] + history
    return llm(messages + [{"role": "user", "content": message}])


print(day1_chat("Who is Lancaster and what is Carllm?", []))

Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm, a leading Insurance Tech company based in San Francisco, California. She has been instrumental in guiding the company since its founding in 2015, focusing on innovative leadership and risk management strategies.

Carllm is an innovative auto insurance product developed by Insurellm. It uses advanced AI algorithms to provide personalized auto insurance solutions, streamlining the process for both insurance companies and their customers. Carllm offers features such as AI-powered risk assessment, instant quoting, customizable coverage plans, fraud detection, and mobile integration, among others.


And the weakness that motivates the rest of the week: ask about someone by first name only, or about something that
isn't a product name, and the lookup finds nothing.

In [4]:
print(get_relevant_context("Who is Avery?"))
print(day1_chat("Who won the IIOTY award in 2023?", []))

[]


I don't have specific information about who won the IIOTY (Insurance Insider's Insurtech Innovation of the Year) award in 2023. This kind of information would typically be available through recent industry publications or the Insurance Insider website. If you need this information, I recommend checking the latest announcements from Insurance Insider or other relevant industry sources.


In [5]:
import gradio as gr

# gr.ChatInterface(day1_chat, type="messages").launch()

## Day 2 – chunks, vectors and Chroma

### Part A: how big is the knowledge base, and how do we cut it up?

The course counts tokens with gpt-4.1-nano's tokenizer; `o200k_base` is that tokenizer, so the number is the same.

In [6]:
documents = fetch_documents()
entire_knowledge_base = "\n\n".join(d["text"] for d in documents)
print(f"{len(documents)} documents, {len(entire_knowledge_base):,} characters")
print(f"{len(tiktoken.get_encoding('o200k_base').encode(entire_knowledge_base)):,} tokens")
print(Counter(d["type"] for d in documents))

76 documents, 304,432 characters


63,555 tokens
Counter({'contracts': 32, 'employees': 32, 'products': 8, 'company': 4})


That's about 65k tokens: it would fit in qwen2.5's context, but only at a big `num_ctx` and slowly - which is the
point of retrieval. Day 2 splits with LangChain's `RecursiveCharacterTextSplitter` into 1000-character chunks with
200 characters of overlap.

In [7]:
chunks = rag_lib.split_documents(documents, chunk_size=1000, chunk_overlap=200)
print(f"Divided into {len(chunks)} chunks")
print(chunks[100])

Divided into 413 chunks
page_content='13. **Climate Risk Analytics:** Forward-looking climate modeling:\n    - IPCC climate scenario analysis (RCP 2.6, 4.5, 8.5)\n    - Transition risk assessment\n    - Physical risk modeling for perils (hurricane, wildfire, flood, drought)\n    - Sea level rise impact analysis\n    - Temperature trend incorporation\n    - Climate-adjusted pricing recommendations\n    - Stranded asset identification\n    - Green reinsurance opportunities\n\n---\n\n## Support\n\nInsurellm commits to comprehensive Enterprise-level support for GlobalRe Partners:\n\n1. **Dedicated Success Team:**\n   - Executive sponsor (CEO-level) with quarterly strategic reviews\n   - Dedicated Senior Vice President of Customer Success with bi-weekly engagement\n   - Technical Account Manager for platform optimization\n   - Solutions Architect team (2 FTE) for strategic initiatives\n   - Catastrophe modeling specialist for analytics support\n   - Quarterly executive business reviews with

### Part B: vectors in Chroma

`all-MiniLM-L6-v2` as in the course: 384 dimensions, runs in a second or two on the Mac. `build_store` reuses the
collection if it already holds these chunks.

In [8]:
start = time.time()
store_day2 = rag_lib.insurellm_store("minilm-1000")
everything = store_day2.everything()
vectors = np.array(everything["embeddings"])
print(f"There are {len(vectors):,} vectors with {vectors.shape[1]:,} dimensions in the vector store ({time.time() - start:.0f}s)")

There are 413 vectors with 384 dimensions in the vector store (0s)


### Part C: visualise

In [9]:
COLOURS = {"products": "blue", "employees": "green", "contracts": "red", "company": "orange"}


def plot_store(store, title, dims=2):
    data = store.everything()
    vectors = np.array(data["embeddings"])
    types = [m["type"] for m in data["metadatas"]]
    reduced = TSNE(n_components=dims, random_state=42).fit_transform(vectors)
    hover = [f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(types, data["documents"])]
    marker = dict(size=5 if dims == 2 else 4, color=[COLOURS[t] for t in types], opacity=0.8)
    if dims == 2:
        trace = go.Scatter(x=reduced[:, 0], y=reduced[:, 1], mode="markers", marker=marker, text=hover, hoverinfo="text")
    else:
        trace = go.Scatter3d(x=reduced[:, 0], y=reduced[:, 1], z=reduced[:, 2], mode="markers", marker=marker, text=hover, hoverinfo="text")
    fig = go.Figure(data=[trace])
    fig.update_layout(title=title, width=850, height=600, margin=dict(r=20, b=10, l=10, t=40))
    fig.show()


plot_store(store_day2, "2D Chroma Vector Store Visualization (MiniLM, 1000-char chunks)")

In [10]:
plot_store(store_day2, "3D Chroma Vector Store Visualization (MiniLM, 1000-char chunks)", dims=3)

## Day 3 – the first real RAG

Retriever + LLM, as in `implementation/answer.py`: embed the question, take the nearest 10 chunks, paste them into
the system prompt.

In [11]:
basic = RAG(store_day2, retrieval_k=10, final_k=10)

for chunk in basic.fetch_context("Who is Avery?")[:3]:
    print(chunk.metadata["source"], "|", chunk.page_content[:90].replace("\n", " "))

knowledge-base/employees/Avery Lancaster.md | ## Other HR Notes - **Professional Development**: Avery has actively participated in leade
knowledge-base/employees/Avery Lancaster.md | - **2022**: **Satisfactory**     Avery focused on rebuilding team dynamics and addressing 
knowledge-base/employees/Avery Lancaster.md | - **2018**: **Exceeds Expectations**     Under Avery’s pivoted vision, Insurellm launched 


In [12]:
answer, _ = basic.answer_question("Who is Averi Lancaster?")
print(answer)

I believe you might have meant to ask about Avery Lancaster. Avery Lancaster is the Co-Founder and Chief Executive Officer (CEO) of Insurellm. She has been instrumental in guiding the company since its founding in 2015, and her innovative leadership has helped Insurellm become a leading provider in the Insurance Tech sector. Prior to founding Insurellm, Avery worked as a Senior Product Manager at Innovate Insurance Solutions, where she developed insurance products for the tech industry.


The course's `app.py` layout - chat on the left, the retrieved context on the right - works with any `RAG` object.
Uncomment to try it; `python week5/local_rag.py` launches the best configuration from day 5 on its own.

In [13]:
# rag_lib.build_chat_ui(basic).launch()

## Day 4 – evaluation

The same 150 test questions as the course (`evaluation/tests.jsonl`), the same metrics: **MRR** (how high the first
chunk containing each keyword ranks), **nDCG** and **keyword coverage** in the top 10, plus an LLM judge scoring the
answers 1–5 for accuracy, completeness and relevance.

In [14]:
tests = rag_lib.load_tests()
example = tests[0]
print(len(tests), "tests")
print(example.question, "|", example.category)
print(example.reference_answer)
print(example.keywords)
Counter(t.category for t in tests)

150 tests
Who won the prestigious IIOTY award in 2023? | direct_fact
Maxine Thompson won the prestigious Insurellm Innovator of the Year (IIOTY) award in 2023.
['Maxine', 'Thompson', 'IIOTY']


Counter({'direct_fact': 70,
         'temporal': 20,
         'spanning': 20,
         'comparative': 10,
         'numerical': 10,
         'relationship': 10,
         'holistic': 10})

In [15]:
rag_lib.evaluate_retrieval(basic, example)

{'mrr': 0.16666666666666666,
 'ndcg': 0.29420493957109245,
 'coverage': 66.66666666666667}

In [16]:
verdict, answer, _ = rag_lib.evaluate_answer(basic, example)
print(answer, "\n")
print(verdict.feedback)
print(verdict.accuracy, verdict.completeness, verdict.relevance)

Maxine Thompson won the prestigious Insurellm IIOTY Innovator Award in 2023. 

The generated answer is correct and matches the reference answer in terms of the winner's name and the year of the award. However, there are slight differences in the phrasing of the award name which could be considered less precise.
5.0 5.0 5.0


### Which chunk size and encoder?

Retrieval metrics only need embeddings, so all 150 questions take seconds per configuration. This is the experiment
behind day 5's "let's use the best chunk size and encoder from yesterday".

In [17]:
retrieval_runs = {}
for variant in ["minilm-1000", "minilm-500", "bge-500"]:
    store = rag_lib.insurellm_store(variant)
    retrieval_runs[variant] = rag_lib.retrieval_report(RAG(store), tests)

summary = pd.DataFrame({name: r[["mrr", "ndcg", "coverage"]].mean() for name, r in retrieval_runs.items()}).T.round(3)
summary

,mrr,ndcg,coverage
minilm-1000,0.739,0.739,90.211
minilm-500,0.751,0.758,91.656
bge-500,0.824,0.806,91.122


In [18]:
pd.DataFrame({name: r.groupby("category")["mrr"].mean() for name, r in retrieval_runs.items()}).round(3)

,minilm-1000,minilm-500,bge-500
category,,,
comparative,0.711,0.685,0.793
direct_fact,0.785,0.843,0.884
holistic,0.673,0.581,0.675
numerical,0.831,0.883,0.875
relationship,0.599,0.742,0.889
spanning,0.584,0.482,0.602
temporal,0.801,0.755,0.871


## Day 5 – advanced RAG

### LLM chunking, exactly as the course does it

The LLM reads a whole document, decides where to cut, and writes a headline, a summary and the original text for each
chunk. On a 14B local model that means generating the document back out - a few minutes per document. Here it is on
one of the shortest files:

In [19]:
shortest = min(documents, key=lambda d: len(d["text"]))
print(shortest["source"], len(shortest["text"]), "characters")

start = time.time()
llm_chunks = rag_lib.llm_chunk_document(shortest)
print(f"{len(llm_chunks)} chunks in {time.time() - start:.0f}s\n")
print(llm_chunks[0].page_content)

knowledge-base/employees/Tyler Brooks.md 1987 characters


4 chunks in 45s

Tyler Brooks HR Record Summary

This chunk contains the summary of Tyler Brooks' HR record, including his date of birth, job title, location, and current salary.

# HR Record

# Tyler Brooks

## Summary
- **Date of Birth:** September 9, 1998
- **Job Title:** Junior Backend Developer
- **Location:** Austin, Texas
- **Current Salary:** $75,000


### The affordable version: enrich deterministic chunks

`enrich_chunks` splits with the text splitter (500 characters, 100 overlap - nothing can be lost or paraphrased) and
asks the LLM only for each chunk's headline and a one-line summary that names who or what it is about. That is the
part that helps retrieval: a chunk from the middle of an HR record that just says "she was promoted" now starts
with *"Maxine Thompson's promotion to Senior Data Engineer"*. It took about 50 minutes for the whole knowledge base
and is cached in `local_db/enriched_chunks.json`.

In [20]:
store_enriched = rag_lib.insurellm_store("bge-enriched")
print(store_enriched.count(), "enriched chunks")
print(store_enriched.everything()["documents"][7])

884 enriched chunks
Join Insurellm: Revolutionizing the Insurance Industry

Discover why Insurellm is a unique place to work, transforming the insurance sector since 2015.

# Careers at Insurellm

## Why Join Insurellm?

At Insurellm, we're not just building software—we're revolutionizing an entire industry. Since our founding in 2015, we've evolved from a high-growth startup to a lean, profitable company with 32 highly talented employees managing 32 active contracts across all eight of our product lines.


In [21]:
plot_store(store_enriched, "2D Chroma Vector Store Visualization (BGE, enriched chunks)")

### Reranking

Retrieve 20 chunks, then reorder them. The course asks the LLM to return the chunk ids in order of relevance;
`rerank_with_llm` does the same (and repairs the list if the model drops or repeats an id). A cross-encoder - a small
model trained to score (question, passage) pairs - does the same job in a fraction of a second.

In [22]:
question = "Who won the IIOTY award?"
chunks = store_enriched.search(question, 10)
for chunk in chunks:
    print(chunk.page_content[:40].replace("\n", " ") + "...")

Maxine Thompson's Additional HR Notes  I...
Priya Sharma's Skills and Feedback  List...
Priya Sharma's Additional Achievements a...
Maxine Thompson's Promotion to Senior Da...
Carlos Rodriguez's Skills and Feedback  ...
Kevin Zhang's Additional HR Notes  Inclu...
Oliver Spencer's Leadership and Mentorsh...
Oliver Spencer's Recent Performance and ...
James Wilson's Patents and Strategic Imp...
Emily Tran's 2023 Performance Review  Hi...


In [23]:
start = time.time()
reranked = rag_lib.rerank_with_llm(question, chunks)
print(f"LLM rerank: {time.time() - start:.1f}s")
for chunk in reranked[:5]:
    print(chunk.page_content[:40].replace("\n", " ") + "...")

LLM rerank: 11.5s
Maxine Thompson's Additional HR Notes  I...
Maxine Thompson's Promotion to Senior Da...
Priya Sharma's Skills and Feedback  List...
Priya Sharma's Additional Achievements a...
Carlos Rodriguez's Skills and Feedback  ...


In [24]:
question = "Who went to Manchester University?"
chunks = store_enriched.search(question, 20)


def manchester_positions(chunks):
    return [i for i, c in enumerate(chunks) if "manchester" in c.page_content.lower()]


print("vector search:  ", manchester_positions(chunks))
start = time.time()
print("LLM rerank:     ", manchester_positions(rag_lib.rerank_with_llm(question, chunks)), f"{time.time() - start:.1f}s")
start = time.time()
print("cross-encoder:  ", manchester_positions(rag_lib.rerank_with_cross_encoder(question, chunks)), f"{time.time() - start:.1f}s")

vector search:   [1]


LLM rerank:      [0] 22.3s


cross-encoder:   [0] 1.9s


### Query rewriting

In [25]:
print(rag_lib.rewrite_query("Who won the IIOTY award?"))
print(rag_lib.rewrite_query("And what does she do now?", [
    {"role": "user", "content": "Who won the IIOTY award in 2023?"},
    {"role": "assistant", "content": "Maxine Thompson won the IIOTY award in 2023."},
]))

What is the winner of the IIOTY award?


What is Maxine Thompson's current occupation?


### Putting it together

`pro_implementation/answer.py`: search with both the original and the rewritten question, merge, rerank, keep the
top 10.

In [26]:
advanced = RAG(store_enriched, rewrite=True, rerank="llm", retrieval_k=20, final_k=10)
answer, chunks = advanced.answer_question("Who went to Manchester University?")
print(advanced.last_query, "\n")
print(answer)

What specific individuals are known to have attended Manchester University? 

Jessica Liu attended the University of Manchester, where she earned a BS in Computer Science.


### Did it help? The evaluation, locally

Retrieval for all 150 questions on each pipeline. The LLM-reranked pipeline makes two LLM calls per question
(rewrite and rerank), so it takes a while.

In [27]:
pipelines = {
    "day 3: MiniLM, 1000 chars": RAG(store_day2),
    "BGE, enriched chunks": RAG(store_enriched),
    "+ rewrite + cross-encoder": RAG(store_enriched, rewrite=True, rerank="cross-encoder", retrieval_k=20),
    "+ rewrite + LLM rerank (day 5)": RAG(store_enriched, rewrite=True, rerank="llm", retrieval_k=20),
}

for name, pipeline in pipelines.items():
    if name not in retrieval_runs:
        retrieval_runs[name] = rag_lib.retrieval_report(pipeline, tests)

order = ["minilm-1000", "minilm-500", "bge-500"] + list(pipelines)[1:]
pd.DataFrame({name: retrieval_runs[name][["mrr", "ndcg", "coverage"]].mean() for name in order}).T.round(3)

,mrr,ndcg,coverage
minilm-1000,0.739,0.739,90.211
minilm-500,0.751,0.758,91.656
bge-500,0.824,0.806,91.122
"BGE, enriched chunks",0.858,0.851,97.200
+ rewrite + cross-encoder,0.916,0.888,96.678
+ rewrite + LLM rerank (day 5),0.926,0.911,97.533


Answer quality needs an answer and a judge call per question - about half a minute each locally - so this runs on
two questions from each category. The judge is qwen2.5:14b too, grading its own answers; with an API key you would
use a different, stronger model as the judge.

In [28]:
sample = rag_lib.sample_tests(tests, per_category=2)
answer_runs = {}
for name in ["day 3: MiniLM, 1000 chars", "+ rewrite + cross-encoder"]:
    answer_runs[name] = rag_lib.answer_report(pipelines[name], sample)

pd.DataFrame({name: r[["accuracy", "completeness", "relevance"]].mean() for name, r in answer_runs.items()}).T.round(2)

,accuracy,completeness,relevance
"day 3: MiniLM, 1000 chars",3.86,3.79,3.50
+ rewrite + cross-encoder,4.29,4.07,3.57


In [29]:
comparison = answer_runs["day 3: MiniLM, 1000 chars"][["question", "category", "accuracy"]].rename(columns={"accuracy": "day 3"})
comparison["advanced"] = answer_runs["+ rewrite + cross-encoder"]["accuracy"].values
comparison

,question,category,day 3,advanced
0,Who won the prestigious IIOTY award in 2023?,direct_fact,5.0,5.0
1,When was Insurellm founded?,direct_fact,5.0,5.0
2,When did Maxine Thompson join Insurellm?,temporal,5.0,5.0
3,When did James Wilson join Insurellm?,temporal,1.0,5.0
4,How much did Priya Sharma's recommendation engine increase conversion by for Marketllm?,comparative,5.0,5.0
5,By what percentage did David Kim reduce deployment time with CI/CD pipelines?,comparative,4.0,5.0
6,How many claims does FastTrack Insurance Services project for year 1?,numerical,5.0,5.0
7,How much annual revenue does Marcus Johnson's portfolio generate?,numerical,5.0,5.0
8,What are the core insurance portals offered by Insurellm?,relationship,5.0,5.0
9,What are the marketplace and infrastructure products offered by Insurellm?,relationship,5.0,5.0


## The week 5 exercise: the Spanish tutor answers from its notes

The tutor from weeks 1–4 now has a knowledge base: grammar and vocabulary notes in `spanish-kb/` (ser vs estar,
por vs para, preterite vs imperfect, gender and articles, the subjunctive, object pronouns, gustar-type verbs,
stem-changing and reflexive verbs, accents, false friends), plus any decks the week 2 tutor saved to `week2/decks/`.
[spanish_tutor_rag.py](spanish_tutor_rag.py) uses what the week found to work:

- chunks are the notes' `##` sections, each prefixed with its note's title - the "headline" idea from day 5 without
  an LLM call, because hand-written notes already have good headings
- BGE embeddings, query rewriting for follow-ups, and the cross-encoder reranker
- answers cite the note they came from, and **Make a deck** turns the notes behind the last answer into a
  Flashcards.com CSV with the week 2 tutor's `save_flashcards`
- its own evaluation set, `spanish_tests.jsonl` (36 questions), scored with the day 4 metrics

In [30]:
import spanish_tutor_rag as tutor

tutor.compare_retrieval()

,pipeline,mrr,ndcg,coverage
0,plain vector search,0.648,0.706,88.889
1,+ query rewrite,0.648,0.706,88.889
2,+ rewrite + cross-encoder rerank,0.904,0.916,95.833


In [31]:
spanish = tutor.tutor_rag()
answer, notes = spanish.answer_question("Should I say 'para el viernes' or 'por el viernes'?")
print(answer)
print("\nNotes used:", [n.metadata["title"] for n in notes])

You should say "para el viernes" (see: Por vs para). This is used to indicate a deadline or a time by which something needs to be done. For example:

- *Necesito el informe para el viernes.* (I need the report by Friday.)

"Por el viernes" would imply something different, like movement through the day or a cause related to Friday, which is not the case here.

Notes used: ['Por vs para', 'Por vs para', 'Por vs para', 'Por vs para', 'Por vs para']


In [32]:
answer, notes = spanish.answer_question("And how do I thank someone for their help?", [
    {"role": "user", "content": "Should I say 'para el viernes' or 'por el viernes'?"},
    {"role": "assistant", "content": answer},
])
print("Searched for:", spanish.last_query, "\n")
print(answer)

Searched for: How do I express gratitude in Spanish for someone's help? 

To thank someone for their help, you can use the phrase "Gracias por tu ayuda" or "Te agradezco tu ayuda." Here are a couple of examples:

- *Gracias por tu ayuda.* (Thank you for your help.)
- *Te agradezco tu ayuda.* (I appreciate your help.)


In [33]:
message, path = tutor.deck_from_notes(notes, count=8)
print(message)
print(path.read_text(encoding="utf-8") if path else "")

TOOL CALLED: save_flashcards('False Friends (Falsos Amigos)', 8 cards)


Saved 8 cards to False_Friends__Falsos_Amigos_.csv. The deck is ready to download and import.
"embarazada","pregnant"
"la librería","the bookshop"
"el éxito","success"
"la carpeta","the folder"
"constipado/a","having a cold"
"actual / actualmente","current / currently"
"la ropa","clothes"
"la decepción","disappointment"



In [34]:
tutor_answers = tutor.evaluate_answers()
print(tutor_answers[["accuracy", "completeness", "relevance"]].mean().round(2))
tutor_answers.groupby("category")[["accuracy", "completeness", "relevance"]].mean().round(2)

accuracy        4.81
completeness    4.33
relevance       3.64
dtype: float64


,accuracy,completeness,relevance
category,,,
comparative,5.00,4.83,4.00
direct_fact,4.72,4.16,3.52
spanning,5.00,4.60,3.80


In [35]:
tutor_answers.sort_values("accuracy").head(5)[["question", "accuracy", "feedback"]]

,question,accuracy,feedback
0,Which verb do I use to say where a party takes place?,1.0,The generated answer is incorrect in suggesting the use of 'dar' or 'tener' to indicate where a party takes place. T...
11,What does 'no quise' mean?,2.0,The generated answer provides a slightly inaccurate translation of 'no quise' by suggesting it means 'I refused' or ...
20,Why do I say 'se lo doy' instead of 'le lo doy'?,5.0,"The generated answer is accurate and provides a clear explanation of the rule, including examples. It also offers ad..."
21,Why does 'leyéndolo' have an accent?,5.0,"The generated answer is accurate and provides a detailed explanation of why 'leyéndolo' has an accent, which is more..."
22,Is it 'me gusta los perros' or 'me gustan los perros'?,5.0,"The generated answer is accurate and provides the correct form of the sentence. However, it could be more thorough i..."


The app: `python week5/spanish_tutor_rag.py` (or uncomment below). Ask a grammar question, look at the notes on the
right, follow up with "and with estar?", then press **Make a deck**.

In [36]:
# tutor.launch()